# 01 · EDA: Sparkov card transactions (`fraudTrain.csv`)
Goal: understand the imbalance, check that fraud behaves differently over time (which justifies the time-based split),
and see which signals separate fraud from legit before modelling. Uses the same cleaned + feature table as `src/train.py`.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.data import load_dataset
from src.evaluate import FIG_DIR
sns.set_theme(style="whitegrid"); FIG_DIR.mkdir(parents=True, exist_ok=True)
df = load_dataset()          # first call builds + caches data/processed/features.parquet (~1 min)
print(df.shape); df.head()

(1296675, 17)


,cc_num,category,amt,gender,city_pop,is_fraud,ts,age,hour,is_night,day_of_week,card_prior_txns,hours_since_prev,n_txn_24h,amt_sum_24h,amt_vs_card_avg,new_category_for_card
0,2703186189652095,misc_net,4.97,F,3495,0,2019-01-01 00:00:18,30.8,0,1,1,0,720.0,0.0,0.0,1.0,1
1,630423337322,grocery_pos,107.23,F,149,0,2019-01-01 00:00:44,40.5,0,1,1,0,720.0,0.0,0.0,1.0,1
2,38859492057661,entertainment,220.11,M,4154,0,2019-01-01 00:00:51,57.0,0,1,1,0,720.0,0.0,0.0,1.0,1
3,3534093764340240,gas_transport,45.00,M,1939,0,2019-01-01 00:01:16,52.0,0,1,1,0,720.0,0.0,0.0,1.0,1
4,375534208663984,misc_pos,41.96,M,99,0,2019-01-01 00:03:06,32.8,0,1,1,0,720.0,0.0,0.0,1.0,1


## 1. Imbalance

In [2]:
rate = df.is_fraud.mean()
print(f"{len(df):,} transactions | {df.is_fraud.sum():,} frauds | fraud rate {rate:.3%} | 1 fraud per {1/rate:.0f} txns")
print(f"Fraud is {df.loc[df.is_fraud==1,'amt'].sum()/df.amt.sum():.1%} of all transaction dollars")
print(f"{df.cc_num.nunique():,} cards | {df.ts.min():%Y-%m-%d} .. {df.ts.max():%Y-%m-%d}")

1,296,675 transactions | 7,506 frauds | fraud rate 0.579% | 1 fraud per 173 txns
Fraud is 4.4% of all transaction dollars
983 cards | 2019-01-01 .. 2020-06-21


## 2. Fraud over time: why the split is chronological

In [3]:
m = df.set_index("ts").resample("MS").agg(txns=("is_fraud", "size"), frauds=("is_fraud", "sum"))
m["fraud_rate_pct"] = m.frauds / m.txns * 100
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(m.index, m.txns, width=20, color="lightgrey", label="transactions")
ax2 = ax.twinx(); ax2.plot(m.index, m.fraud_rate_pct, "o-", c="C3", label="fraud rate %")
for d in ["2020-01-01", "2020-04-01"]: ax.axvline(pd.Timestamp(d), c="k", ls="--")
ax.set_title("Monthly volume and fraud rate (dashed = train | val | test boundaries)")
ax.set_ylabel("transactions"); ax2.set_ylabel("fraud rate (%)")
fig.tight_layout(); fig.savefig(FIG_DIR / "eda_fraud_over_time.png", dpi=150); m

,txns,frauds,fraud_rate_pct
ts,,,
2019-01-01,52525,506,0.963351
2019-02-01,49866,517,1.036779
2019-03-01,70939,494,0.696373
2019-04-01,68078,376,0.552308
2019-05-01,72532,408,0.562510
2019-06-01,86064,354,0.411322
2019-07-01,86596,331,0.382235
2019-08-01,87359,382,0.437276
2019-09-01,70652,418,0.591632


In [4]:
# Fraud comes in bursts on the same card: frauds per card and how many days they span
fc = df[df.is_fraud == 1].groupby("cc_num").ts.agg(["size", "min", "max"])
fc["span_days"] = (fc["max"] - fc["min"]).dt.total_seconds() / 86400
print(f"{len(fc)} cards ever defrauded ({len(fc)/df.cc_num.nunique():.0%} of cards)")
fc[["size", "span_days"]].describe().round(1)

762 cards ever defrauded (78% of cards)


,size,span_days
count,762.0,762.0
mean,9.9,1.7
std,3.0,0.4
min,2.0,0.0
25%,8.0,1.5
50%,10.0,1.9
75%,12.0,1.9
max,19.0,2.9


## 3. Amount, time of day, category

In [5]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for lbl, g in df.groupby("is_fraud"):
    ax[0].hist(np.log10(g.amt.clip(lower=1)), bins=60, density=True, alpha=.6, label=["legit", "fraud"][lbl])
ax[0].set_xlabel("log10(amount $)"); ax[0].set_title("Amount distribution"); ax[0].legend()
h = df.groupby("hour").is_fraud.mean() * 100
ax[1].bar(h.index, h.values, color=["C3" if (x >= 22 or x < 4) else "C0" for x in h.index])
ax[1].set_xlabel("hour of day"); ax[1].set_ylabel("fraud rate (%)"); ax[1].set_title("Fraud rate by hour (red = night)")
fig.tight_layout(); fig.savefig(FIG_DIR / "eda_amount_hour.png", dpi=150)

In [6]:
cat = df.groupby("category").agg(txns=("is_fraud", "size"), fraud_rate=("is_fraud", "mean"),
                                 fraud_amt=("amt", lambda s: s[df.loc[s.index, "is_fraud"] == 1].sum()))
cat = cat.sort_values("fraud_rate")
fig, ax = plt.subplots(figsize=(7, 4.5)); ax.barh(cat.index, cat.fraud_rate * 100)
ax.set_xlabel("fraud rate (%)"); ax.set_title("Fraud rate by merchant category")
fig.tight_layout(); fig.savefig(FIG_DIR / "eda_category.png", dpi=150)
cat.sort_values("fraud_amt", ascending=False).assign(fraud_rate=lambda d: (d.fraud_rate*100).round(2))

,txns,fraud_rate,fraud_amt
category,,,
shopping_net,97543,1.76,1711723.71
shopping_pos,116672,0.72,739245.09
misc_net,63287,1.45,729266.76
grocery_pos,123638,1.41,543797.90
entertainment,94014,0.25,117323.79
misc_pos,79655,0.31,54571.02
home,123115,0.16,50971.66
food_dining,91461,0.17,18131.62
gas_transport,131659,0.47,7594.11


## 4. Card-history features (built from each card's past only)

In [7]:
feats = ["amt_vs_card_avg", "n_txn_24h", "amt_sum_24h", "hours_since_prev", "new_category_for_card", "age"]
df.groupby("is_fraud")[feats].median().T.rename(columns={0: "legit (median)", 1: "fraud (median)"}).round(2)

is_fraud,legit (median),fraud (median)
amt_vs_card_avg,0.66,5.16
n_txn_24h,3.00,4.00
amt_sum_24h,171.14,1674.16
hours_since_prev,4.62,1.40
new_category_for_card,0.00,0.00
age,44.00,47.80


In [8]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, f, clip in zip(axes, ["amt_vs_card_avg", "n_txn_24h", "hours_since_prev"], [20, 15, 72]):
    sns.boxplot(data=df.assign(v=df[f].clip(upper=clip)), x="is_fraud", y="v", ax=ax, showfliers=False)
    ax.set_title(f); ax.set_xlabel("is_fraud"); ax.set_ylabel(f"(clipped at {clip})")
fig.tight_layout(); fig.savefig(FIG_DIR / "eda_card_history.png", dpi=150)

## 5. Takeaways
- **Imbalance:** 7,506 frauds in 1.30M transactions (0.58%, 1 in 173), but fraud is 4.4% of all transaction dollars: frauds are large. Accuracy is meaningless here (approve-all scores 99.4%).
- **Time:** the monthly fraud rate swings between 0.38% and 1.04%, so the data is not stationary. Fraud arrives in **bursts**: 762 of 983 cards (78%) are hit, with a median of 10 frauds packed into ~1.9 days. A random split would put the same burst in train and test; the split is chronological.
- **Strongest raw signals:** amount (fraud medians are far larger), night hours (22:00-04:00), and three categories: `shopping_net` (1.76% fraud, $1.7M of fraud), `misc_net` (1.45%) and `grocery_pos` (1.41%).
- **Card-history signals:** the median fraud is 5.2x the card's usual amount (legit: 0.66x), comes 1.4h after the previous transaction (legit: 4.6h), and pushes 24h spend to $1,674 (legit: $171). Fraud is a change in the card's behaviour, which is what the past-only features capture.
- **Consequences for modelling:** PR-AUC and $-recall instead of accuracy/ROC-AUC; chronological split; class weighting/SMOTE compared rather than assumed; thresholds chosen on cost and analyst capacity.